# Alternative Models — LSTM, Gaussian Process, Neural Network

The main solution uses gradient boosting. This notebook answers a fair
question: *the data has dates — is it a time series, and would sequence
models (RNN/LSTM) or Gaussian Process regression work better?*

**Is this time series data?** Only partly. A classic time series is *one
entity observed repeatedly in order* (one stock's daily price, one sensor's
readings). Here we have **48,000 independent loads** — every row is a
different shipment on a different lane. The only true "series" hiding inside
is the *market level* (average price drifting with the seasons), and the
main model already captures it through date features and `market_index`.
This kind of data is called **tabular / panel data with a time component**.

Below, each alternative model is trained on the same time split as the main
notebook (train Jan–Aug, holdout Sep–Oct) so the numbers are directly
comparable.

In [ ]:
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("scripts")
from train_model import clean, lane_city_stats, build_features, make_model

from sklearn.metrics import (mean_absolute_error,
                             mean_absolute_percentage_error, r2_score)

def evaluate(name, y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    print(f"{name:<30} MAE ${mean_absolute_error(y_true, y_pred):7.2f}  "
          f"RMSE ${np.sqrt(np.mean((y_true - y_pred) ** 2)):7.2f}  "
          f"MAPE {mean_absolute_percentage_error(y_true, y_pred):6.2%}  "
          f"R2 {r2_score(y_true, y_pred):.4f}")

dev = clean(pd.read_csv("data/train_test.csv", parse_dates=["date"]), is_train=True)
cutoff = pd.Timestamp("2025-09-01")
tr, ho = dev[dev.date < cutoff], dev[dev.date >= cutoff]
stats = lane_city_stats(tr)
x_tr, x_ho = build_features(tr, stats), build_features(ho, stats)
y_tr, y_ho = tr.posted_rate.to_numpy(), ho.posted_rate.to_numpy()
print(f"train {len(tr):,} | holdout {len(ho):,}")

## Reference: the main gradient boosting model

In [ ]:
t0 = time.time()
gbm = make_model().fit(x_tr, y_tr)
evaluate("gradient boosting", y_ho, gbm.predict(x_ho))
print(f"training time: {time.time() - t0:.1f}s")

## 1. Gaussian Process Regression

GPR is a beautiful model (it even gives uncertainty estimates), but exact
GPR must build and invert an **n × n covariance matrix** — cost grows as
**O(n³)**. With our 38,000 training rows that matrix alone would need
~11 GB and the inversion would take days. In practice GPR is used below
~5,000–10,000 points (or with sparse approximations that trade accuracy).

To still give it a fair look, we train it on a **2,000-row random sample**
with a standard RBF + noise kernel on scaled features.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
idx = rng.choice(len(x_tr), 2000, replace=False)

scaler = StandardScaler().fit(x_tr)
xs_tr = scaler.transform(x_tr)[idx]
xs_ho = scaler.transform(x_ho)
ys = y_tr[idx]
y_mean, y_std = ys.mean(), ys.std()

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0) \
         + WhiteKernel(noise_level=0.1)
t0 = time.time()
gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=False, random_state=42, n_restarts_optimizer=0)
gpr.fit(xs_tr, (ys - y_mean) / y_std)
pred_gpr = gpr.predict(xs_ho) * y_std + y_mean
evaluate("gaussian process (2k sample)", y_ho, pred_gpr)
print(f"training time on only 2,000 rows: {time.time() - t0:.1f}s")

Even on this small sample the training already takes minutes — and it can
only use 8% of the available data, which costs accuracy. That is the
practical reason GPR is not the right tool for 48k tabular rows.

## 2. Neural network (MLP) on the same features — GPU-ready

A feed-forward network is the *correct* deep-learning shape for this data
(one row in → one price out). The cell below auto-detects a GPU: on a
machine with CUDA it prints `device: cuda` and trains there; on CPU it
still finishes in a couple of minutes.

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

xs_tr_t = torch.tensor(scaler.transform(x_tr), dtype=torch.float32)
xs_ho_t = torch.tensor(scaler.transform(x_ho), dtype=torch.float32)
ys_t = torch.tensor((y_tr - y_mean) / y_std, dtype=torch.float32).unsqueeze(1)

model = nn.Sequential(
    nn.Linear(xs_tr_t.shape[1], 256), nn.ReLU(), nn.Dropout(0.1),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 1),
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)
loss_fn = nn.L1Loss()
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(xs_tr_t, ys_t), batch_size=1024, shuffle=True)

t0 = time.time()
for epoch in range(40):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss_fn(model(xb), yb).backward()
        opt.step()
    sched.step()
print(f"trained 40 epochs in {time.time() - t0:.1f}s")

model.eval()
with torch.no_grad():
    pred_mlp = model(xs_ho_t.to(device)).cpu().numpy().ravel() * y_std + y_mean
evaluate("neural network (MLP)", y_ho, pred_mlp)

The MLP gets respectably close but does not beat gradient boosting — the
usual outcome on medium-sized tabular data, where tree ensembles remain
state of the art.

## 3. LSTM — used where it actually fits: the daily market series

An LSTM expects an **ordered sequence of the same thing**. Feeding it
independent loads makes no sense (their order in the file is arbitrary).
The honest way to use an LSTM here is on the one real series we have:
**the daily average rate per mile**. We train it to read the previous 28
days and predict the next day, then check it on Sep–Oct.

In [ ]:
daily = (dev.assign(rpm=dev.posted_rate / dev.distance)
            .groupby("date").rpm.mean().asfreq("D").interpolate())
d_mean, d_std = daily[daily.index < cutoff].mean(), daily[daily.index < cutoff].std()
z = (daily - d_mean) / d_std

WINDOW = 28
xs, ys2, dates = [], [], []
vals = z.to_numpy(dtype=np.float32)
for i in range(WINDOW, len(vals)):
    xs.append(vals[i - WINDOW:i]); ys2.append(vals[i]); dates.append(z.index[i])
xs = torch.tensor(np.array(xs)).unsqueeze(-1)
ys2 = torch.tensor(np.array(ys2)).unsqueeze(-1)
dates = pd.DatetimeIndex(dates)
is_tr = dates < cutoff

class LSTMForecaster(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 64, num_layers=2, batch_first=True, dropout=0.1)
        self.head = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])

lstm = LSTMForecaster().to(device)
opt = torch.optim.Adam(lstm.parameters(), lr=3e-3)
t0 = time.time()
for epoch in range(300):
    lstm.train(); opt.zero_grad()
    loss = nn.functional.l1_loss(lstm(xs[is_tr].to(device)), ys2[is_tr].to(device))
    loss.backward(); opt.step()
print(f"trained LSTM in {time.time() - t0:.1f}s (device: {device})")

lstm.eval()
with torch.no_grad():
    pred_daily = lstm(xs[~is_tr].to(device)).cpu().numpy().ravel() * d_std + d_mean
actual_daily = daily[dates[~is_tr]]

plt.figure(figsize=(10, 3.5))
plt.plot(actual_daily.index, actual_daily.values, label="actual daily avg $/mile")
plt.plot(actual_daily.index, pred_daily, label="LSTM forecast")
plt.legend(); plt.title("LSTM on the market-level series (Sep-Oct)")
plt.tight_layout(); plt.show()
mae_pct = np.mean(np.abs(pred_daily - actual_daily.values) / actual_daily.values)
print(f"daily-average forecast error: {mae_pct:.2%}")

The LSTM tracks the market curve reasonably — but notice what it predicts:
**one number per day** (the market average). It cannot price an individual
load, because it never sees distance, lane, equipment, or weight. To use it
for this assessment we would still need a per-load model on top; the
calendar + `market_index` features inside the gradient boosting model
already do that job more simply.

## Conclusion

| Model | Fits this data? | Holdout result |
|---|---|---|
| Gradient boosting | ✅ built for tabular rows | best (MAE ≈ $77, R² ≈ 0.96) |
| Neural net (MLP) | ✅ valid, GPU-ready | close, but behind GBM |
| Gaussian Process | ⚠️ O(n³) — max a few thousand rows | worse (can't use all data) |
| LSTM / RNN | ⚠️ only for the market-level series | forecasts the average, not loads |

Gradient boosting stays the final model: best accuracy, fastest training,
and no GPU required.